# Bab 1 — Introduction to Clustering Evaluation

Notebook pendamping Bab 1 Level 5.

Notebook ini mengikuti halaman bab di situs. Jalankan sel dari atas ke bawah.

## Learning Objectives

Setelah mempelajari bab ini, kamu mampu:

- Menjelaskan apa itu *clustering evaluation* dan mengapa hasil clustering perlu dinilai.
- Menjelaskan mengapa evaluasi clustering berbeda dari evaluasi klasifikasi.
- Menyebutkan ciri hasil clustering yang baik: *compactness*, *separation*, *density*, *consistency*, dan *stability*.
- Membedakan tiga pendekatan: *internal*, *external*, dan *relative evaluation*.
- Menjelaskan mengapa gambar saja tidak cukup, dan mengapa skor saja juga tidak cukup.
- Menghitung tiga skor pertama dengan scikit-learn: Silhouette, Davies-Bouldin, dan Calinski-Harabasz.
- Mengenali bahwa algoritma clustering **selalu** menghasilkan kelompok, bahkan pada data yang tidak punya kelompok.

## Langkah 0: Pindah ke folder utama proyek

In [ ]:
import os
from pathlib import Path

if not Path("data").exists():
    os.chdir("../..")

## Bagian A: Siapkan alat

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.preprocessing import StandardScaler

## Bagian B: Tiga dataset untuk dibandingkan

In [ ]:
# Tiga dataset dengan 300 titik dan 3 pusat yang sama.
# Bedanya hanya cluster_std: seberapa lebar sebaran tiap kelompok.
X_clear, _ = make_blobs(n_samples=300, centers=3, cluster_std=0.6, random_state=42)
X_overlap, _ = make_blobs(n_samples=300, centers=3, cluster_std=3.0, random_state=42)

# Data tanpa kelompok alami: titik tersebar rata di sebuah persegi
rng = np.random.default_rng(42)
X_uniform = rng.uniform(low=-10, high=10, size=(300, 2))

datasets = {
    "Kelompok jelas": X_clear,
    "Kelompok tumpang tindih": X_overlap,
    "Tanpa kelompok alami": X_uniform,
}
for name, X in datasets.items():
    print(f"{name:25s} bentuk data: {X.shape}")

## Bagian C: Menjalankan K-Means pada tiap dataset

In [ ]:
labels_by_name = {}
for name, X in datasets.items():
    model = KMeans(n_clusters=3, n_init=10, random_state=42)
    labels_by_name[name] = model.fit_predict(X)
    print(f"{name:25s} ukuran kelompok: {np.bincount(labels_by_name[name]).tolist()}")

## Bagian D: Tiga skor pertama

In [ ]:
rows = []
for name, X in datasets.items():
    labels = labels_by_name[name]
    rows.append(
        {
            "Dataset": name,
            "Silhouette": silhouette_score(X, labels),
            "Davies-Bouldin": davies_bouldin_score(X, labels),
            "Calinski-Harabasz": calinski_harabasz_score(X, labels),
        }
    )
score_table = pd.DataFrame(rows).set_index("Dataset")
print(score_table.round(3))

## Bagian E: Menggambar tiga dataset

In [ ]:
palette = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#D55E00", "#56B4E9"]

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for ax, (name, X) in zip(axes, datasets.items()):
    labels = labels_by_name[name]
    for j in range(3):
        ax.scatter(X[labels == j, 0], X[labels == j, 1], s=14, color=palette[j])
    sil = score_table.loc[name, "Silhouette"]
    ax.set_title(f"{name}\nSilhouette = {sil:.3f}")
    ax.set_xlabel("Fitur 1")
    ax.set_ylabel("Fitur 2")
plt.tight_layout()
plt.show()

## Bagian F: Evaluasi relatif: membandingkan beberapa nilai K

In [ ]:
# Evaluasi relatif: jalankan K-Means dengan K berbeda, bandingkan skornya
k_values = range(2, 8)
rel_rows = []
for k in k_values:
    labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X_clear)
    rel_rows.append({"K": k, "Silhouette": silhouette_score(X_clear, labels)})
rel_table = pd.DataFrame(rel_rows).set_index("K")
print(rel_table.round(3))
best_k = rel_table["Silhouette"].idxmax()
print("K dengan Silhouette tertinggi:", best_k)

## Bagian G: Data tanpa kelompok alami

In [ ]:
# Data tanpa kelompok alami juga mendapat skor, dan K-Means tetap menghasilkan kelompok
uniform_rows = []
for k in range(2, 6):
    labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X_uniform)
    uniform_rows.append({"K": k, "Silhouette": silhouette_score(X_uniform, labels)})
print(pd.DataFrame(uniform_rows).set_index("K").round(3))

## Bagian H: Menggambar skor tiap K

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(rel_table.index, rel_table["Silhouette"], marker="o", color=palette[0], label="Kelompok jelas")
uniform_table = pd.DataFrame(uniform_rows).set_index("K")
ax.plot(uniform_table.index, uniform_table["Silhouette"], marker="s", color=palette[1], label="Tanpa kelompok alami")
ax.set_xlabel("Jumlah kelompok (K)")
ax.set_ylabel("Silhouette Score")
ax.set_title("Skor Silhouette untuk tiap nilai K")
ax.legend()
plt.tight_layout()
plt.show()

## Bagian I: Evaluasi pada Mall Customers

In [ ]:
df = pd.read_csv("data/mall_customers.csv")
features = ["Annual Income (k$)", "Spending Score (1-100)"]
X_mall = StandardScaler().fit_transform(df[features])

labels_mall = KMeans(n_clusters=5, n_init=10, random_state=42).fit_predict(X_mall)
print("Ukuran kelompok:", np.bincount(labels_mall).tolist())
print(f"Silhouette        : {silhouette_score(X_mall, labels_mall):.3f}")
print(f"Davies-Bouldin    : {davies_bouldin_score(X_mall, labels_mall):.3f}")
print(f"Calinski-Harabasz : {calinski_harabasz_score(X_mall, labels_mall):.1f}")

## Latihan

1. Pada `X_clear`, jalankan K-Means dengan K = 2, 3, dan 4. Berapa Davies-Bouldin untuk masing-masing? Pada K berapa nilainya paling rendah?
2. Ganti `cluster_std=3.0` menjadi `4.5` pada `X_overlap`. Apa yang terjadi pada Silhouette?
3. Ubah `random_state` pada `np.random.default_rng` menjadi 7. Apakah K terbaik pada data acak berubah? Apa artinya?
4. Jalankan K-Means pada Mall Customers dengan K = 4 dan K = 6. Bandingkan Silhouette dengan K = 5 (0,555).
5. Jelaskan dengan kata-katamu mengapa kita tidak bisa memakai *accuracy* untuk menilai clustering.
6. Mengapa kelompok "0" pada satu eksekusi tidak boleh dianggap sama dengan kelompok "0" pada eksekusi lain?

Jawaban lengkap ada di halaman bab.

## Ringkasan

- *Clustering evaluation* menilai seberapa baik hasil clustering, padahal tidak ada label benar saat belajar.
- Clustering yang baik punya kelompok **kompak** dan **terpisah**. Kepadatan, keseragaman, dan kestabilan juga penting.
- Tiga pendekatan: **internal** (tanpa label benar), **external** (dengan label benar), dan **relative** (membandingkan beberapa hasil).
- Algoritma clustering **selalu** menghasilkan kelompok, bahkan pada data acak. Silhouette pada data acak kita masih 0,401.
- Gambar dan skor saling melengkapi. Jangan memutuskan hanya dari satu angka.
- Fungsi metrik di scikit-learn menerima data dan label, dan tidak melatih apa pun.